# Faza 2.2: dotreniravanje BERTić-a za NER (Halo beba)

**Pre pokretanja:** `Runtime → Change runtime type → T4 GPU`.

**Ulaz:** folder `data_ner/` koji pravi `ner_priprema.py` (`train.jsonl`, `dev.jsonl`, `test.jsonl`, `labels.json`, `split.json`).
Lokalno ga spakuj: `zip -r data_ner.zip data_ner` i otpremi u ćeliji ispod.

**Napomena o podacima:** otpremanjem u Colab tekst dijaloga ide na Google servere. Proveri sa mentorom da li je to dozvoljeno za ovu bazu.

**Sadržaj:**
1. Instalacija i učitavanje podataka
2. Bazni model (rečnik iz trening skupa), za poređenje
3. Dotreniravanje BERTić-a na fiksnoj podeli (105 / 15 / 30 dijaloga)
4. Rezultati po tipu entiteta i analiza grešaka
5. Unakrsna validacija u 5 foldova (glavni rezultat za tezu, traje duže)

## 1. Instalacija i podaci

In [ ]:
!pip install -q seqeval

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NEMA GPU - uključi T4 u Runtime podešavanjima")

In [ ]:
# Otpremi data_ner.zip (napravljen lokalno sa: zip -r data_ner.zip data_ner)
from google.colab import files
files.upload()
!unzip -o -q data_ner.zip
!ls data_ner

# Alternativa: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/data_ner .

In [ ]:
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data_ner")
MODEL_NAME = "classla/bcms-bertic"

# Podešavanja treniranja
LEARNING_RATE = 3e-5
EPOCHS = 10
BATCH_SIZE = 16
SEED = 42

records = [json.loads(line) for name in ["train", "dev", "test"]
           for line in open(DATA / f"{name}.jsonl", encoding="utf-8")]
label_list = json.loads((DATA / "labels.json").read_text(encoding="utf-8"))
split = json.loads((DATA / "split.json").read_text(encoding="utf-8"))

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}


def select(dialog_ids):
    ids = set(dialog_ids)
    return [r for r in records if r["dialog_id"] in ids]


train_rows, dev_rows, test_rows = select(split["train"]), select(split["dev"]), select(split["test"])
print(f"Replika: train {len(train_rows)}, dev {len(dev_rows)}, test {len(test_rows)}; oznaka: {len(label_list)}")

## 2. Bazni model: rečnik iz trening skupa

Svi entiteti iz trening skupa se zapamte (tekst → najčešći tip), pa se u test tekstu traže najduži pogoci.
Ovo pokazuje koliko se postiže samim pamćenjem; BERTić treba da bude bolji, posebno na oblicima koje nije video.

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.scheme import IOB2


def spans_of(tokens, tags):
    """Entiteti iz BIO oznaka kao (početak, kraj, tip)."""
    spans, start, label = [], None, None
    for i, tag in enumerate(tags + ["O"]):
        if start is not None and not tag.startswith("I-"):
            spans.append((start, i, label))
            start = None
        if tag.startswith("B-"):
            start, label = i, tag[2:]
    return spans


def build_dictionary(rows):
    votes = {}
    for r in rows:
        words = [t.lower() for t in r["tokens"]]
        for s, e, label in spans_of(r["tokens"], r["ner_tags"]):
            votes.setdefault(tuple(words[s:e]), Counter())[label] += 1
    return {k: v.most_common(1)[0][0] for k, v in votes.items()}


def dictionary_predict(dictionary, tokens, max_len=8):
    words = [t.lower() for t in tokens]
    tags, i = ["O"] * len(tokens), 0
    while i < len(tokens):
        for n in range(min(max_len, len(tokens) - i), 0, -1):  # najduži pogodak prvo
            label = dictionary.get(tuple(words[i:i + n]))
            if label:
                tags[i:i + n] = ["B-" + label] + ["I-" + label] * (n - 1)
                i += n
                break
        else:
            i += 1
    return tags


def scores(gold, pred):
    return {"precision": precision_score(gold, pred, mode="strict", scheme=IOB2),
            "recall": recall_score(gold, pred, mode="strict", scheme=IOB2),
            "f1": f1_score(gold, pred, mode="strict", scheme=IOB2)}


dictionary = build_dictionary(train_rows)
gold_test = [r["ner_tags"] for r in test_rows]
baseline_pred = [dictionary_predict(dictionary, r["tokens"]) for r in test_rows]
baseline = scores(gold_test, baseline_pred)
print(f"Rečnik: {len(dictionary)} izraza")
print("Bazni model (test):", {k: round(v, 3) for k, v in baseline.items()})

## 3. Dotreniravanje BERTić-a

In [ ]:
from datasets import Dataset
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorForTokenClassification(tokenizer)


def encode(rows):
    """Deli reči na delove; oznaku dobija samo prvi deo svake reči, ostali -100 (ignorišu se)."""
    enc = tokenizer([r["tokens"] for r in rows], is_split_into_words=True,
                    truncation=True, max_length=512)
    all_labels = []
    for i, r in enumerate(rows):
        labels, previous = [], None
        for word_id in enc.word_ids(i):
            if word_id is None or word_id == previous:
                labels.append(-100)
            else:
                labels.append(label2id[r["ner_tags"][word_id]])
            previous = word_id
        all_labels.append(labels)
    enc["labels"] = all_labels
    return Dataset.from_dict(dict(enc))


def to_tags(logits, label_ids):
    """Iz izlaza modela vraća BIO oznake po reči (samo pozicije prvih delova reči)."""
    pred_ids = np.argmax(logits, axis=-1)
    gold, pred = [], []
    for p_row, l_row in zip(pred_ids, label_ids):
        keep = l_row != -100
        gold.append([id2label[i] for i in l_row[keep]])
        pred.append([id2label[i] for i in p_row[keep]])
    return gold, pred


def compute_metrics(eval_pred):
    gold, pred = to_tags(eval_pred.predictions, eval_pred.label_ids)
    return scores(gold, pred)


def train_and_test(train_rows, dev_rows, test_rows, out_dir, seed=SEED):
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        # zagrevanje: prvih 10% koraka learning rate raste od 0 (ceo broj radi u svim verzijama)
        warmup_steps=int(0.1 * EPOCHS * -(-len(train_rows) // BATCH_SIZE)),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,     # zadržava epohu sa najboljim F1 na dev skupu
        metric_for_best_model="f1",
        seed=seed,
        report_to="none",
        logging_steps=50,
    )
    trainer = Trainer(model=model, args=args, train_dataset=encode(train_rows),
                      eval_dataset=encode(dev_rows), data_collator=collator,
                      processing_class=tokenizer, compute_metrics=compute_metrics)
    trainer.train()
    output = trainer.predict(encode(test_rows))
    gold, pred = to_tags(output.predictions, output.label_ids)
    return trainer, gold, pred

In [ ]:
trainer, gold, pred = train_and_test(train_rows, dev_rows, test_rows, out_dir="bertic_ner")
bertic = scores(gold, pred)
print("Bazni model:", {k: round(v, 3) for k, v in baseline.items()})
print("BERTić:     ", {k: round(v, 3) for k, v in bertic.items()})

## 4. Rezultati po tipu i analiza grešaka

In [ ]:
report = classification_report(gold, pred, mode="strict", scheme=IOB2, output_dict=True)
base_report = classification_report(gold_test, baseline_pred, mode="strict", scheme=IOB2, output_dict=True)
per_type = pd.DataFrame({
    "broj_u_testu": {t: report[t]["support"] for t in report if t.isupper()},
    "F1_recnik": {t: base_report.get(t, {}).get("f1-score", 0.0) for t in report if t.isupper()},
    "precision": {t: report[t]["precision"] for t in report if t.isupper()},
    "recall": {t: report[t]["recall"] for t in report if t.isupper()},
    "F1_bertic": {t: report[t]["f1-score"] for t in report if t.isupper()},
}).sort_values("broj_u_testu", ascending=False).round(3)
per_type

In [ ]:
def error_types(rows, gold, pred):
    """Svrstava svaku grešku: pogrešan tip, pogrešne granice, propušten ili lažan entitet."""
    counts, examples = Counter(), []
    for r, g, p in zip(rows, gold, pred):
        gs, ps = set(spans_of(r["tokens"], g)), set(spans_of(r["tokens"], p))
        for s, e, lab in gs - ps:
            same_span = [x for x in ps if x[:2] == (s, e)]
            overlap = [x for x in ps if x[0] < e and x[1] > s]
            if same_span:
                kind = f"pogrešan tip ({lab} → {same_span[0][2]})"
            elif overlap:
                kind = f"pogrešne granice ({lab})"
            else:
                kind = f"propušten ({lab})"
            counts[kind] += 1
            examples.append((kind, " ".join(r["tokens"][s:e]), " ".join(r["tokens"])))
        for s, e, lab in ps - gs:
            if not any(x[0] < e and x[1] > s for x in gs):
                counts[f"lažan ({lab})"] += 1
    return counts, examples


counts, examples = error_types(test_rows, gold, pred)
print("Najčešće greške:")
for kind, n in counts.most_common(20):
    print(f"  {n:>4}  {kind}")

In [ ]:
# Primeri grešaka (promeni filter da vidiš određeni tip greške)
for kind, span, sentence in [x for x in examples if "pogrešan tip" in x[0]][:10]:
    print(f"{kind}: »{span}«\n    {sentence}\n")

In [ ]:
# Čuvanje predikcija za kasniju analizu i gradnju grafa iz NER-a
with open("predikcije_test.jsonl", "w", encoding="utf-8") as f:
    for r, g, p in zip(test_rows, gold, pred):
        f.write(json.dumps({**r, "pred_tags": p}, ensure_ascii=False) + "\n")
per_type.to_csv("rezultati_po_tipu.csv")
print("Sačuvano: predikcije_test.jsonl, rezultati_po_tipu.csv")

## 5. Unakrsna validacija (5 foldova)

Svaki fold od 30 dijaloga je jednom test skup; od ostalih 120, 15 ide u dev, a 105 u trening.
Rezultat je prosek ± standardna devijacija, pouzdaniji od jedne podele na malom skupu.
Na T4 GPU traje otprilike 5 puta duže od sekcije 3.

In [ ]:
cv_rows = []
for k, test_ids in enumerate(split["folds"]):
    rest = [d for d in sorted({r["dialog_id"] for r in records}) if d not in set(test_ids)]
    random.Random(SEED + k).shuffle(rest)
    dev_ids, train_ids = rest[:15], rest[15:]
    tr, dv, te = select(train_ids), select(dev_ids), select(test_ids)

    _, g, p = train_and_test(tr, dv, te, out_dir=f"cv_fold{k}")
    base_p = [dictionary_predict(build_dictionary(tr), r["tokens"]) for r in te]
    rep = classification_report(g, p, mode="strict", scheme=IOB2, output_dict=True)
    row = {"fold": k, "F1_bertic": f1_score(g, p, mode="strict", scheme=IOB2),
           "F1_recnik": f1_score(g, base_p, mode="strict", scheme=IOB2)}
    row.update({f"F1_{t}": rep[t]["f1-score"] for t in rep if t.isupper()})
    cv_rows.append(row)
    print(f"Fold {k}: BERTić F1 = {row['F1_bertic']:.3f}, rečnik F1 = {row['F1_recnik']:.3f}")

cv = pd.DataFrame(cv_rows)
print(f"\nBERTić: {cv['F1_bertic'].mean():.3f} ± {cv['F1_bertic'].std():.3f}")
print(f"Rečnik: {cv['F1_recnik'].mean():.3f} ± {cv['F1_recnik'].std():.3f}")

In [ ]:
type_cols = [c for c in cv.columns if c.startswith("F1_") and c not in ("F1_bertic", "F1_recnik")]
cv_per_type = pd.DataFrame({"F1_prosek": cv[type_cols].mean(), "F1_std": cv[type_cols].std()})
cv_per_type.index = [c[3:] for c in cv_per_type.index]
cv_per_type = cv_per_type.sort_values("F1_prosek", ascending=False).round(3)
cv.to_csv("cv_po_foldu.csv", index=False)
cv_per_type.to_csv("cv_po_tipu.csv")
cv_per_type

## 6. Preuzimanje rezultata i modela

In [ ]:
# Rezultati (mali fajlovi)
files.download("rezultati_po_tipu.csv")
files.download("predikcije_test.jsonl")
# files.download("cv_po_tipu.csv"); files.download("cv_po_foldu.csv")

# Model (~450 MB) - najbolje sačuvati na Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# trainer.save_model("/content/drive/MyDrive/bertic_ner_halo_beba")
# tokenizer.save_pretrained("/content/drive/MyDrive/bertic_ner_halo_beba")